# Instance-Based Music Recommendation System (K-Nearest Neighbours)

### 1. Mathematical Backbone & Approach
Each song $i$ in our world dataset of $N$ songs is represented as a binary feature vector $\mathbf{x}_i \in \{0, 1\}^D$, where $D$ is the total number of artist and genre features dynamically inferred from the dataset:
$$\mathbf{x}_i = [a_1, a_2, \dots, a_n, g_1, g_2, \dots, g_m]^T$$

Given a dynamic queue $Q = \{s_1, s_2, \dots, s_q\}$ of $q$ song names, our system retrieves their corresponding feature vectors and constructs a **User Taste Profile Vector (Queue Centroid)** $\mathbf{u} \in \mathbb{R}^D$:
$$\mathbf{u} = \sum_{j=1}^{q} w_j \mathbf{x}_{s_j}, \quad \text{where } \sum_{j=1}^{q} w_j = 1$$

To recommend the next $K=5$ songs, we compute the **Cosine Distance** (or Euclidean / Jaccard distance) between the queue centroid $\mathbf{u}$ and every candidate song $\mathbf{x}_i \notin Q$:
$$d_{\text{cos}}(\mathbf{u}, \mathbf{x}_i) = 1 - \frac{\mathbf{u} \cdot \mathbf{x}_i}{\Vert{}\mathbf{u}\Vert{}_2 \Vert{}\mathbf{x}_i\Vert{}_2}$$
We also handle matrix sparsity using `scipy.sparse.csr_matrix` so only non-zero binary entries are stored and multiplied.

In [8]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.sparse import csr_matrix

DATA_PATH = "PES1UG25AM517_I_Saniya_Data.csv"

# Load dataset and verify
world_df = pd.read_csv(DATA_PATH)
print(f"Loaded World Dataset with shape: {world_df.shape}")
display(world_df.head(5))

Loaded World Dataset with shape: (126, 28)


,Song_Name,Queen,ABBA,Britney Spears,The Weeknd,Eagles,Nirvana,Beyoncé,Daft Punk,Led Zeppelin,...,Pearl Jam,Classic Rock,Hard Rock,Metal,Grunge / Alt Rock,Pop,Dance Pop,R&B,Alt R&B,Electronic/EDM
0,Bohemian Rhapsody - Queen,1,0,0,0,0,0,0,0,0,...,0,1,1,0,0,1,0,0,0,0
1,Don't Stop Me Now - Queen,1,0,0,0,0,0,0,0,0,...,0,1,1,0,0,1,0,0,0,0
2,Another One Bites the Dust - Queen,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,1,1,0,0,0
3,We Will Rock You - Queen,1,0,0,0,0,0,0,0,0,...,0,1,1,0,0,1,0,0,0,0
4,Somebody to Love - Queen,1,0,0,0,0,0,0,0,0,...,0,1,1,0,0,1,0,0,0,0


In [9]:
class DynamicKNNMusicRecommender:
    """
    Instance-based K-Nearest Neighbours (KNN) Music Recommendation System.
    Fully dynamic: works on any dataset where Column 0 is Song Name and Columns 1..M are binary features.
    """
    def __init__(self, dataframe: pd.DataFrame):
        if dataframe is None or dataframe.empty or dataframe.shape[1] < 2:
            raise ValueError("Invalid dataset: Must contain 1 song column and >= 1 feature columns.")

        self.df = dataframe.copy()

        # Dynamically infer song title column and feature columns (Zero hardcoded indices/names)
        self.song_col = self.df.columns[0]
        self.feature_cols = list(self.df.columns[1:])

        # Normalize whitespace in song names
        self.df[self.song_col] = self.df[self.song_col].astype(str).str.strip()
        self.df = self.df.drop_duplicates(subset=[self.song_col]).reset_index(drop=True)

        # Dense Feature Matrix X (N x D) and Sparse CSR Matrix for handling sparsity
        self.X_dense = self.df[self.feature_cols].to_numpy(dtype=np.float64)
        self.X_sparse = csr_matrix(self.X_dense)

        # Compute Sparsity Percentage
        self.sparsity_pct = (1.0 - (self.X_sparse.nnz / self.X_dense.size)) * 100.0

        # Fast O(1) lookup dictionaries
        self.song_to_idx = {song: idx for idx, song in enumerate(self.df[self.song_col])}
        self.lower_to_exact = {song.lower(): song for song in self.df[self.song_col]}

        # Active user queue (stores song name strings)
        self.queue = []

    def _resolve_song_name(self, song_name: str) -> str:
        """Resolves a song name string against the world dataset (case-insensitive fallback)."""
        if not isinstance(song_name, str) or not song_name.strip():
            return None
        cleaned = song_name.strip()
        if cleaned in self.song_to_idx:
            return cleaned
        if cleaned.lower() in self.lower_to_exact:
            return self.lower_to_exact[cleaned.lower()]
        return None

    def get_feature_vector(self, song_name: str) -> np.ndarray:
        """Dynamically retrieves the 1D binary feature vector for a given song name string."""
        resolved = self._resolve_song_name(song_name)
        if resolved is None:
            raise KeyError(f"Song '{song_name}' does not exist in the world dataset.")
        return self.X_dense[self.song_to_idx[resolved]]

    def initialize_queue(self, seed_queue: list):
        """Initializes the listening queue from a list of seed song strings."""
        self.queue = []
        for song in seed_queue:
            self.append_song(song, verbose=False)
        print(f"Queue initialized with {len(self.queue)} valid songs | Matrix Sparsity: {self.sparsity_pct:.2f}%")

    def recommend(self, k: int = 5, metric: str = "cosine", recency_weighted: bool = True) -> pd.DataFrame:
        """Recommends the top K nearest songs not currently in the queue."""
        if len(self.queue) == 0:
            print("[Edge Case Warning] The queue is currently empty! Cannot compute recommendations.")
            return pd.DataFrame()

        # Retrieve feature vectors for all songs in the active queue
        queue_vectors = np.array([self.get_feature_vector(s) for s in self.queue])

        # Compute weights (Recency-weighted vs Uniform average)
        if recency_weighted and len(self.queue) > 1:
            weights = np.linspace(0.5, 1.5, len(self.queue))
            weights = weights / np.sum(weights)
        else:
            weights = np.ones(len(self.queue)) / len(self.queue)

        # User Profile Centroid Vector u (1 x D)
        centroid_u = np.average(queue_vectors, axis=0, weights=weights).reshape(1, -1)

        # Compute distances between Centroid u and World Matrix X
        if metric == "cosine":
            u_norm = np.linalg.norm(centroid_u)
            x_norms = np.linalg.norm(self.X_dense, axis=1)
            dot_products = self.X_sparse.dot(centroid_u.flatten())
            denom = np.where(u_norm * x_norms == 0, 1e-10, u_norm * x_norms)
            similarities = dot_products / denom
            distances = 1.0 - similarities
        elif metric == "euclidean":
            distances = cdist(centroid_u, self.X_dense, metric="euclidean").flatten()
            similarities = 1.0 / (1.0 + distances)
        elif metric == "jaccard":
            binary_u = (centroid_u > 0).astype(np.float64)
            distances = cdist(binary_u, self.X_dense, metric="jaccard").flatten()
            similarities = 1.0 - distances
        else:
            raise ValueError(f"Unsupported metric '{metric}'.")

        # Tie-breaker: Minimum item-to-item distance to any individual song in the queue
        item_to_item_dists = cdist(queue_vectors, self.X_dense, metric="cosine")
        closest_queue_song_dist = np.min(item_to_item_dists, axis=0)

        # Exclude songs already in the queue
        in_queue_set = set(self.queue)
        candidate_indices = [
            idx for idx, title in enumerate(self.df[self.song_col])
            if title not in in_queue_set
        ]

        if len(candidate_indices) == 0:
            print("[Edge Case Warning] Every song in the dataset is already in your queue!")
            return pd.DataFrame()

        # Sort candidates by smallest distance
        ranked_indices = sorted(
            candidate_indices,
            key=lambda i: (round(distances[i], 6), round(closest_queue_song_dist[i], 6))
        )
        top_k_idx = ranked_indices[:min(k, len(ranked_indices))]

        return pd.DataFrame({
            "Rank": range(1, len(top_k_idx) + 1),
            "Song_Name": self.df.iloc[top_k_idx][self.song_col].values,
            f"Distance ({metric})": np.round(distances[top_k_idx], 4),
            "Similarity_Score": np.round(similarities[top_k_idx], 4)
        })

    # ==================== DYNAMIC QUEUE MANIPULATION METHODS ====================

    def append_song(self, song_name: str, verbose: bool = True) -> bool:
        """Appends a song to the end of the queue."""
        resolved = self._resolve_song_name(song_name)
        if resolved is None:
            print(f"[Edge Case Warning] '{song_name}' not found in the world dataset. Ignored.")
            return False
        if resolved in self.queue:
            print(f"[Edge Case Notice] '{resolved}' is already in the queue. Duplicate skipped.")
            return False
        self.queue.append(resolved)
        if verbose:
            print(f"[+] Appended '{resolved}' to queue (Total songs: {len(self.queue)}).")
        return True

    def insert_song(self, index: int, song_name: str) -> bool:
        """Inserts a song at a specific index inside the queue."""
        resolved = self._resolve_song_name(song_name)
        if resolved is None:
            print(f"[Edge Case Warning] '{song_name}' not found in the world dataset.")
            return False
        if resolved in self.queue:
            print(f"[Edge Case Notice] '{resolved}' is already in the queue.")
            return False
        safe_idx = max(0, min(index, len(self.queue)))
        self.queue.insert(safe_idx, resolved)
        print(f"[+] Inserted '{resolved}' at position {safe_idx} (Total songs: {len(self.queue)}).")
        return True

    def delete_song(self, song_name: str) -> bool:
        """Deletes a specific song by name from the queue."""
        resolved = self._resolve_song_name(song_name)
        if resolved not in self.queue:
            print(f"[Edge Case Warning] '{song_name}' is not in the current queue.")
            return False
        self.queue.remove(resolved)
        print(f"[-] Removed '{resolved}' from queue (Total songs: {len(self.queue)}).")
        return True

    def delete_at_index(self, index: int) -> bool:
        """Deletes a song at a given 0-based position in the queue."""
        if not (0 <= index < len(self.queue)):
            print(f"[Edge Case Warning] Index {index} out of bounds for queue of length {len(self.queue)}.")
            return False
        removed = self.queue.pop(index)
        print(f"[-] Removed '{removed}' at index {index} (Total songs: {len(self.queue)}).")
        return True

    def clear_and_autogenerate_queue(self, starter_song: str, length: int = 10, metric: str = "cosine"):
        """Deletes the full queue and builds a brand-new queue purely from recommendations."""
        resolved = self._resolve_song_name(starter_song)
        if resolved is None:
            print(f"[Edge Case Warning] Starter song '{starter_song}' not found in dataset.")
            return
        self.queue = [resolved]
        print(f"[*] Cleared queue. Auto-generating {length}-song queue seeded with '{resolved}'...")
        while len(self.queue) < length:
            next_rec = self.recommend(k=1, metric=metric, recency_weighted=True)
            if next_rec.empty:
                break
            self.queue.append(next_rec.iloc[0]["Song_Name"])
        print(f"[*] Completed auto-generated queue of {len(self.queue)} songs!")

    def show_queue(self):
        """Displays the current active queue."""
        print(f"\n--- Current Listening Queue ({len(self.queue)} songs) ---")
        if not self.queue:
            print("(Empty Queue)")
        for idx, s in enumerate(self.queue, 1):
            print(f"  {idx:2d}. {s}")
        print("-" * 46)

In [10]:
# Instantiate the recommender on the loaded World Dataset
recommender = DynamicKNNMusicRecommender(world_df)

# 1. Initialize with the 10-song Seed Queue (strings only)
seed_queue = [
    "Hotel California - Eagles",
    "Call Out My Name - The Weeknd",
    "Bohemian Rhapsody - Queen",
    "Toxic - Britney Spears",
    "Smells Like Teen Spirit - Nirvana",
    "Halo - Beyoncé",
    "One More Time - Daft Punk",
    "Stairway to Heaven - Led Zeppelin",
    "Dancing Queen - ABBA",
    "Enter Sandman - Metallica"
]

recommender.initialize_queue(seed_queue)
recommender.show_queue()

# 2. Generate Initial Top 5 Recommendations
print("\n>>> INITIAL TOP 5 RECOMMENDATIONS (Cosine KNN):")
initial_recs = recommender.recommend(k=5, metric="cosine")
display(initial_recs)

Queue initialized with 10 valid songs | Matrix Sparsity: 87.77%

--- Current Listening Queue (10 songs) ---
   1. Hotel California - Eagles
   2. Call Out My Name - The Weeknd
   3. Bohemian Rhapsody - Queen
   4. Toxic - Britney Spears
   5. Smells Like Teen Spirit - Nirvana
   6. Halo - Beyoncé
   7. One More Time - Daft Punk
   8. Stairway to Heaven - Led Zeppelin
   9. Dancing Queen - ABBA
  10. Enter Sandman - Metallica
----------------------------------------------

>>> INITIAL TOP 5 RECOMMENDATIONS (Cosine KNN):


,Rank,Song_Name,Distance (cosine),Similarity_Score
0,1,Break My Soul - Beyoncé,0.3216,0.6784
1,2,Another One Bites the Dust - Queen,0.3309,0.6691
2,3,Get Lucky - Daft Punk,0.3343,0.6657
3,4,Don't Stop Me Now - Queen,0.3442,0.6558
4,5,We Will Rock You - Queen,0.3442,0.6558


In [11]:
# Scenario A: Append the #1 recommended song to the queue
top_recommended_song = initial_recs.iloc[0]["Song_Name"]
recommender.append_song(top_recommended_song)

print("\n>>> UPDATED RECOMMENDATIONS AFTER APPENDING TOP RECOMMENDATION:")
display(recommender.recommend(k=5, metric="cosine"))

# Scenario B: User switches mood to Heavy Metal / Hard Rock (Append, Insert, Delete)
print("\n>>> USER MOOD SHIFT: Adding Metal/Hard Rock tracks & removing Dance Pop...")
recommender.append_song("Master of Puppets - Metallica")
recommender.append_song("Paranoid - Black Sabbath")
recommender.insert_song(len(recommender.queue) - 1, "In the End - Linkin Park")
recommender.delete_song("Dancing Queen - ABBA")

print("\n>>> DYNAMICALLY UPDATED RECOMMENDATIONS (Shifted toward Rock/Metal):")
display(recommender.recommend(k=5, metric="cosine"))

[+] Appended 'Break My Soul - Beyoncé' to queue (Total songs: 11).

>>> UPDATED RECOMMENDATIONS AFTER APPENDING TOP RECOMMENDATION:


,Rank,Song_Name,Distance (cosine),Similarity_Score
0,1,Crazy in Love - Beyoncé,0.2314,0.7686
1,2,Single Ladies - Beyoncé,0.2314,0.7686
2,3,Cuff It - Beyoncé,0.2314,0.7686
3,4,Formation - Beyoncé,0.2314,0.7686
4,5,Irreplaceable - Beyoncé,0.2314,0.7686



>>> USER MOOD SHIFT: Adding Metal/Hard Rock tracks & removing Dance Pop...
[+] Appended 'Master of Puppets - Metallica' to queue (Total songs: 12).
[+] Appended 'Paranoid - Black Sabbath' to queue (Total songs: 13).
[+] Inserted 'In the End - Linkin Park' at position 12 (Total songs: 14).
[-] Removed 'Dancing Queen - ABBA' from queue (Total songs: 13).

>>> DYNAMICALLY UPDATED RECOMMENDATIONS (Shifted toward Rock/Metal):


,Rank,Song_Name,Distance (cosine),Similarity_Score
0,1,Nothing Else Matters - Metallica,0.2637,0.7363
1,2,Iron Man - Black Sabbath,0.3033,0.6967
2,3,War Pigs - Black Sabbath,0.3033,0.6967
3,4,N.I.B. - Black Sabbath,0.3033,0.6967
4,5,Children of the Grave - Black Sabbath,0.3033,0.6967


In [12]:
# Scenario C: Delete full queue and build a 6-song queue purely from recommendations
recommender.clear_and_autogenerate_queue(starter_song="Blinding Lights - The Weeknd", length=6)
recommender.show_queue()
print("\n>>> NEXT 5 RECOMMENDATIONS FOR AUTO-GENERATED R&B/POP QUEUE:")
display(recommender.recommend(k=5, metric="cosine"))

# Scenario D: Edge Case Detection Tests
print("\n>>> TESTING EDGE CASES:")
recommender.append_song("NonExistent Song - Unknown Artist")  # Song not in dataset
recommender.append_song("Blinding Lights - The Weeknd")       # Duplicate song already in queue
recommender.delete_song("Hotel California - Eagles")          # Deleting song not in queue
recommender.delete_at_index(99)                               # Out-of-bounds index deletion

# Scenario E: Compare Cosine vs. Euclidean vs. Jaccard Distance
for metric_name in ["cosine", "euclidean", "jaccard"]:
    print(f"\n=== Top 5 Recommendations using {metric_name.upper()} Distance ===")
    display(recommender.recommend(k=5, metric=metric_name))

[*] Cleared queue. Auto-generating 6-song queue seeded with 'Blinding Lights - The Weeknd'...
[*] Completed auto-generated queue of 6 songs!

--- Current Listening Queue (6 songs) ---
   1. Blinding Lights - The Weeknd
   2. Can't Feel My Face - The Weeknd
   3. Call Out My Name - The Weeknd
   4. The Hills - The Weeknd
   5. Save Your Tears - The Weeknd
   6. After Hours - The Weeknd
----------------------------------------------

>>> NEXT 5 RECOMMENDATIONS FOR AUTO-GENERATED R&B/POP QUEUE:


,Rank,Song_Name,Distance (cosine),Similarity_Score
0,1,Die For You - The Weeknd,0.0105,0.9895
1,2,Starboy - The Weeknd,0.0955,0.9045
2,3,Halo - Beyoncé,0.3986,0.6014
3,4,Stay - Rihanna,0.3986,0.6014
4,5,Crazy in Love - Beyoncé,0.4271,0.5729



>>> TESTING EDGE CASES:
[Edge Case Warning] 'NonExistent Song - Unknown Artist' not found in the world dataset. Ignored.
[Edge Case Notice] 'Blinding Lights - The Weeknd' is already in the queue. Duplicate skipped.
[Edge Case Warning] 'Hotel California - Eagles' is not in the current queue.
[Edge Case Warning] Index 99 out of bounds for queue of length 6.

=== Top 5 Recommendations using COSINE Distance ===


,Rank,Song_Name,Distance (cosine),Similarity_Score
0,1,Die For You - The Weeknd,0.0105,0.9895
1,2,Starboy - The Weeknd,0.0955,0.9045
2,3,Halo - Beyoncé,0.3986,0.6014
3,4,Stay - Rihanna,0.3986,0.6014
4,5,Crazy in Love - Beyoncé,0.4271,0.5729



=== Top 5 Recommendations using EUCLIDEAN Distance ===


,Rank,Song_Name,Distance (euclidean),Similarity_Score
0,1,Die For You - The Weeknd,0.2949,0.7723
1,2,Starboy - The Weeknd,0.9593,0.5104
2,3,Halo - Beyoncé,1.6392,0.3789
3,4,Stay - Rihanna,1.6392,0.3789
4,5,Needed Me - Rihanna,1.7570,0.3627



=== Top 5 Recommendations using JACCARD Distance ===


,Rank,Song_Name,Distance (jaccard),Similarity_Score
0,1,Starboy - The Weeknd,0.1667,0.8333
1,2,Die For You - The Weeknd,0.3333,0.6667
2,3,Break My Soul - Beyoncé,0.4286,0.5714
3,4,Crazy in Love - Beyoncé,0.5714,0.4286
4,5,Single Ladies - Beyoncé,0.5714,0.4286


### 2. Algorithm Comparison & Why Our KNN Method is Rudimentary

While our K-Nearest Neighbours model demonstrates the mathematical backbone of vector spaces, distance metrics, and centroid profiling using Linear Algebra, it is rudimentary compared to production systems like Spotify for several reasons:

1. **Orthogonal Binary Vectors vs. Dense Latent Embeddings**:
   In one-hot encoding, every artist column is orthogonal ($\mathbf{a}_i \cdot \mathbf{a}_j = 0$ for $i \neq j$). Our model only connects two different artists if they share coarse genre columns. Production systems use Matrix Factorization (SVD) and Deep Learning embeddings to project songs into dense low-dimensional spaces ($\mathbb{R}^d$) where similar artists naturally cluster together.
2. **Lack of Collaborative Filtering**:
   Our instance-based system is purely content-based—it only looks at static artist/genre tags of the current queue. Modern platforms use Collaborative Filtering across millions of user listening histories to learn non-obvious song relationships without relying on manual genre labels.
3. **Absence of Acoustic & Sequential Modeling**:
   Averaging queue vectors into a single centroid loses exact track ordering and raw audio features (tempo, energy, key, valence). Production systems use sequential deep learning models and contextual bandits to balance exploitation (matching current mood) with exploration (discovering new genres).